# Systematic Cross-Asset Skewness Strategy (v3, Modular)

This notebook is a **clean execution canvas**: each section orchestrates one stage of the strategy while heavy logic lives in reusable Python modules under `experiments/cam_skewness_core/`.

The workflow follows an institutional research process:
1. Universe and data quality checks
2. Signal engineering and exploratory diagnostics
3. Portfolio construction and execution mapping
4. Class/global performance and risk diagnostics
5. Attribution and practical implementation checks


## 1. Environment and Modular Imports

This section loads all reusable building blocks. The notebook stays concise by calling tested functions instead of embedding long implementation blocks.


In [ ]:
import warnings
import sys
from pathlib import Path

# Make project imports robust regardless of where the notebook kernel starts.
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "experiments").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "experiments").exists():
    raise RuntimeError(f"Could not locate project root from cwd={Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from IPython.display import display

from experiments.cam_skewness_core.config import (
    LOOKBACK,
    HISTORY_DAYS,
    VOL_TARGET,
    UNIVERSE,
    FACTOR_TICKERS,
    SAMPLE_TICKERS,
    SAMPLE_WEIGHT_TICKERS,
)
from experiments.cam_skewness_core.data_loader import (
    LOADER_VERSION,
    load_universe_yf,
    universe_summary,
    data_quality_summary,
)
from experiments.cam_skewness_core.signal import (
    add_skew_features,
    skew_distribution_by_class,
    build_monthly_signal_table,
    add_rank_weights,
)
from experiments.cam_skewness_core.backtest import (
    apply_monthly_weights_to_daily,
    compute_asset_class_returns,
    compute_global_factor,
    weight_sanity_checks,
    monthly_turnover,
)
from experiments.cam_skewness_core.analytics import (
    run_asset_class_alpha_beta,
    run_equity_factor_regression,
    build_coef_table,
    performance_table,
)
from experiments.cam_skewness_core.plots import (
    plot_sample_returns_and_skew,
    plot_skew_distribution_box,
    plot_latest_skew_and_weights,
    plot_sample_weight_history,
    plot_asset_class_cum_returns,
    plot_asset_class_strategy_vs_market,
    plot_global_factor,
    plot_global_drawdown,
    plot_turnover_by_class,
)

warnings.filterwarnings('ignore', category=FutureWarning)
pd.options.display.float_format = '{:.6f}'.format
plt.rcParams['figure.dpi'] = 120
print('Project root on path:', PROJECT_ROOT)



## 2. Strategy Definition and Universe Integrity

This section fixes core hyperparameters and validates universe structure. From a PM perspective, this is where you lock your research contract: horizon, risk target, and investable set.


In [ ]:
print('LOOKBACK:', LOOKBACK)
print('HISTORY_DAYS:', HISTORY_DAYS)
print('VOL_TARGET:', VOL_TARGET)
print('Loader:', LOADER_VERSION)

univ = universe_summary(UNIVERSE)
display(univ)
print('Total listed tickers:', int(univ['TickersListed'].sum()))
print('Total unique tickers:', int(univ['TickersUnique'].sum()))


## 3. Data Ingestion and Pre-Trade Data QA

The objective here is not just to pull prices, but to prove data reliability before signal generation: schema validity, failure transparency, and coverage diagnostics.


In [ ]:
all_data, failures = load_universe_yf(UNIVERSE, HISTORY_DAYS)
print('all_data shape:', all_data.shape)
print('columns:', list(all_data.columns))

if all_data.empty:
    raise RuntimeError('No data loaded from yfinance. Check internet access and package version.')
if 'Ticker' not in all_data.columns:
    raise RuntimeError(f"Schema mismatch after load: {list(all_data.columns)}")

display(all_data.head())

if not failures.empty:
    print(f'Failed tickers: {len(failures)}')
    display(failures.head(20))
else:
    print('No ticker failures during load.')


### 3.1 Data Coverage Diagnostics (Trader-Facing)

These diagnostics answer practical questions:
- How balanced is history across assets?
- Are any symbols under-covered or sparse?
- Does each asset class have stable breadth over time?


In [ ]:
quality = data_quality_summary(all_data)
display(quality.head(15))

class_snapshot = all_data.groupby('AssetClass', as_index=False).agg(
    Tickers=('Ticker', 'nunique'),
    Obs=('Date', 'count'),
    Start=('Date', 'min'),
    End=('Date', 'max'),
)
display(class_snapshot.sort_values('AssetClass'))

# Better breadth diagnostics for mostly-stable universes:
# 1) summary stats by class
# 2) min/median/max utilization plot
# 3) yearly utilization heatmap
breadth = all_data.groupby(['Date', 'AssetClass'])['Ticker'].nunique().reset_index(name='ActiveTickers')
expected = univ[['AssetClass', 'TickersUnique']].rename(columns={'TickersUnique': 'ExpectedTickers'})
breadth = breadth.merge(expected, on='AssetClass', how='left')
breadth['UtilizationPct'] = 100.0 * breadth['ActiveTickers'] / breadth['ExpectedTickers']

breadth_stats = breadth.groupby('AssetClass', as_index=False).agg(
    ExpectedTickers=('ExpectedTickers', 'first'),
    MinActive=('ActiveTickers', 'min'),
    MedianActive=('ActiveTickers', 'median'),
    MaxActive=('ActiveTickers', 'max'),
    MinUtilizationPct=('UtilizationPct', 'min'),
    MedianUtilizationPct=('UtilizationPct', 'median'),
    MaxUtilizationPct=('UtilizationPct', 'max'),
    FullCoveragePct=('UtilizationPct', lambda s: 100.0 * (s == 100.0).mean()),
)

breadth_stats = breadth_stats.sort_values('AssetClass').reset_index(drop=True)
display(breadth_stats.round(2))

# Plot A: min/median/max active tickers by class
plot_df = breadth_stats.copy()
y = np.arange(len(plot_df))

plt.figure(figsize=(11, 5))
plt.hlines(y=y, xmin=plot_df['MinActive'], xmax=plot_df['MaxActive'], color='steelblue', linewidth=3, label='Min-Max range')
plt.scatter(plot_df['MedianActive'], y, color='navy', s=70, zorder=3, label='Median active')
plt.yticks(y, plot_df['AssetClass'])
plt.xlabel('Active Tickers')
plt.title('Universe Breadth by Asset Class (Min / Median / Max)')
plt.legend(loc='lower right')
plt.grid(axis='x', alpha=0.2)
plt.show()

# Plot B: yearly average utilization heatmap (Active / Expected)
breadth['Year'] = breadth['Date'].dt.year
heat = breadth.groupby(['AssetClass', 'Year'])['UtilizationPct'].mean().unstack('Year').sort_index()

fig, ax = plt.subplots(figsize=(12, 4.5))
im = ax.imshow(heat.values, aspect='auto', vmin=0, vmax=100, cmap='YlGnBu')
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=0)
ax.set_title('Yearly Universe Utilization Heatmap (%)')
ax.set_xlabel('Year')
ax.set_ylabel('Asset Class')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Utilization %')

# annotate cells for readability
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = heat.iat[i, j]
        ax.text(j, i, f'{val:.0f}', ha='center', va='center', color='black', fontsize=8)

plt.tight_layout()
plt.show()


## 4. Signal Engineering and EDA

This section builds the rolling skew signal and pressure-tests its behavior. The focus is both statistical correctness and interpretability under real market regimes.


In [ ]:
all_data = add_skew_features(all_data, LOOKBACK)
print('signal-ready shape:', all_data.shape)
display(all_data.head())


In [ ]:
dist_tbl = skew_distribution_by_class(all_data)
display(dist_tbl)
plot_sample_returns_and_skew(all_data, SAMPLE_TICKERS)
plot_skew_distribution_box(all_data)


## 5. Portfolio Construction and Execution Mapping

This section converts signal into tradable weights:
- end-of-month signal extraction
- next-day activation (anti-look-ahead)
- long-short normalization within each asset class
- daily forward-fill of live weights


In [ ]:
monthly_vals = build_monthly_signal_table(all_data)
monthly_vals = add_rank_weights(monthly_vals)

display(monthly_vals.head())

latest_cross_section = monthly_vals.sort_values('Date').groupby(['Date', 'AssetClass']).size().tail(5)
display(latest_cross_section)


### 5.1 Cross-Sectional Signal Inspection (v1-Inspired)

These views are trader-critical. They show whether the latest month-end cross-section is economically sensible:
- which commodities are extreme on skew,
- which names receive long vs short risk budget.


In [ ]:
commodity_snapshot = plot_latest_skew_and_weights(monthly_vals, asset_class='Commodities')
display(commodity_snapshot[['Ticker', 'EOMSkew', 'SkewWeight']])


### 5.2 Weight Path Diagnostics (v1-Inspired)

Weight trajectories for representative ETFs reveal regime shifts, signal instability, and implementation smoothness over time.


In [ ]:
plot_sample_weight_history(monthly_vals, SAMPLE_WEIGHT_TICKERS)


In [ ]:
all_data_weights = apply_monthly_weights_to_daily(all_data, monthly_vals)
asset_portfolios = compute_asset_class_returns(all_data_weights)

display(all_data_weights[['Date', 'Ticker', 'AssetClass', 'LogReturn', 'SkewWeight', 'SkewWeightFF']].head())
display(asset_portfolios.head())


### 5.3 Asset-Class Performance Lens

We inspect strategy performance by sleeve first. This helps isolate where the signal is monetized and where it under-delivers before global aggregation.


In [ ]:
plot_asset_class_cum_returns(asset_portfolios)
plot_asset_class_strategy_vs_market(asset_portfolios, ncols=3)


## 6. Global Portfolio Construction and Risk Lens

Class sleeves are volatility-normalized to create a balanced global factor. We also include PM-style diagnostics: drawdown and compact performance stats.


In [ ]:
asset_portfolios_scaled, gcf = compute_global_factor(
    asset_portfolios,
    lookback=LOOKBACK,
    vol_target=VOL_TARGET,
    scale_market_with_strategy_vol=False,
)

plot_global_factor(gcf)
plot_global_drawdown(gcf)

global_perf = pd.DataFrame({
    'Portfolio': ['GlobalSkew', 'GlobalMarket'],
    'Obs': [len(gcf), len(gcf)],
    'AnnReturn': [gcf['Return'].mean() * 252, gcf['MktReturn'].mean() * 252],
    'AnnVol': [gcf['Return'].std(ddof=0) * np.sqrt(252), gcf['MktReturn'].std(ddof=0) * np.sqrt(252)],
})
global_perf['Sharpe'] = global_perf['AnnReturn'] / global_perf['AnnVol']
display(global_perf.round(4))


In [ ]:
class_perf = performance_table(
    returns_df=asset_portfolios,
    date_col='Date',
    ret_col='PortfolioReturn',
    group_col='AssetClass',
)
display(class_perf.round(4))


### 6.1 Capacity/Cost Awareness via Turnover

Turnover is a practical proxy for implementation drag. We summarize one-way monthly turnover by sleeve and run simple cost scenarios on the global factor.


In [ ]:
turnover = monthly_turnover(monthly_vals)
display(turnover.groupby('AssetClass', as_index=False)['Turnover'].agg(['mean', 'median', 'max']).reset_index())
plot_turnover_by_class(turnover)

global_turnover = turnover.groupby('Date', as_index=False)['Turnover'].mean()
gcf_cost = gcf.set_index('Date').copy()

for bps in [5, 10, 20]:
    cost = pd.Series(0.0, index=gcf_cost.index)
    tmp = global_turnover.set_index('Date')['Turnover']
    ix = cost.index.intersection(tmp.index)
    cost.loc[ix] = tmp.loc[ix] * (bps / 10000.0)
    gcf_cost[f'Return_net_{bps}bps'] = gcf_cost['Return'] - cost

plt.figure(figsize=(12, 5))
plt.plot(gcf_cost.index, gcf_cost['Return'].cumsum(), label='Gross')
plt.plot(gcf_cost.index, gcf_cost['Return_net_5bps'].cumsum(), label='Net 5bps')
plt.plot(gcf_cost.index, gcf_cost['Return_net_10bps'].cumsum(), label='Net 10bps')
plt.plot(gcf_cost.index, gcf_cost['Return_net_20bps'].cumsum(), label='Net 20bps')
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.title('Global Skew Factor: Gross vs Cost-Adjusted Scenarios')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend()
plt.show()


## 7. Attribution: Market and Style Exposures

Attribution helps answer whether returns come from true skew premia or hidden beta/factor tilts.


In [ ]:
results_table = run_asset_class_alpha_beta(asset_portfolios)
display(results_table.round(6))


In [ ]:
equity_model, equity_reg_df = run_equity_factor_regression(
    asset_portfolios,
    factor_tickers=FACTOR_TICKERS,
    history_days=HISTORY_DAYS,
)
print(equity_model.summary())


In [ ]:
coef_table = build_coef_table(equity_model)
display(coef_table)


## 8. Implementation Validation and Operational Checks

These checks confirm portfolio construction is behaving exactly as intended and remains audit-ready.


In [ ]:
sanity = weight_sanity_checks(monthly_vals)
display(sanity.describe(include='all'))

assert (sanity['long_sum'].round(10) == 1.0).all(), 'Long side does not sum to +1 for all periods.'
assert (sanity['short_sum'].round(10) == -1.0).all(), 'Short side does not sum to -1 for all periods.'
assert (sanity['net'].round(10) == 0.0).all(), 'Net exposure is not zero for all periods.'
print('Weight sanity checks passed: long=+1, short=-1, net=0 across all dates/classes.')


## 9. PM/Trader Interpretation Notes

- The strategy is implemented as a **cross-sectional, class-neutral relative value process**, not a directional market bet.
- The strongest sleeves can dominate global PnL; class-level diagnostics should guide allocation overlays.
- Turnover and cost scenarios should always be reviewed before claiming deployability.
- Attribution confirms whether apparent alpha is independent or mostly explained by market/style exposures.

This notebook is intentionally modular so you can iterate on assumptions (lookback, universe, execution/cost model, scaling choice) with minimal notebook edits.
